In [1]:

import os, warnings, json, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")
%matplotlib inline

In [3]:
BASE="/working/outputs" if os.path.isdir("/working") else "outputs"
EDA,RESULT,METRIC=[os.path.join(BASE,x) for x in ("eda","results","metrics")]
for d in (EDA,RESULT,METRIC): os.makedirs(d,exist_ok=True)

def mape(y,yh):
    y,yh=np.asarray(y,float),np.asarray(yh,float); m=y>0
    return np.mean(np.abs((y[m]-yh[m])/y[m]))*100
def wape(y,yh):
    y,yh=np.asarray(y,float),np.asarray(yh,float)
    return np.sum(np.abs(y-yh))/np.sum(np.abs(y))*100
def smape(y,yh):
    y,yh=np.asarray(y,float),np.asarray(yh,float); d=np.abs(y)+np.abs(yh); m=d!=0
    return np.mean(2*np.abs(y[m]-yh[m])/d[m])*100
METRIC_NAMES=["MAE","RMSE","MAPE","WAPE","sMAPE","R2"]
def metrics(y,yh):
    return (mean_absolute_error(y,yh),np.sqrt(mean_squared_error(y,yh)),
            mape(y,yh),wape(y,yh),smape(y,yh),r2_score(y,yh))

In [5]:
# !uv pip install datasets

In [8]:
# 1. Load FreshRetailNet-50K from Hugging Face
from datasets import load_dataset

ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K", split="train")
# drop the hourly sequence columns: hours_sale sums to the target (leakage) and both are heavy
ds = ds.remove_columns([c for c in ["hours_sale","hours_stock_status"] if c in ds.column_names])
df = ds.to_pandas()
del ds
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

DatasetNotFoundError: Dataset 'Dingdong-Inc/FreshRetailNet-5K' doesn't exist on the Hub or cannot be accessed.

In [ ]:
# 2. Sample stores + aggregate to store x category x day
# The raw data is 4.5M rows. We sample a set of stores and aggregate products up to the first category level per store per day, which keeps the data real but makes the target larger and more stable
